<a href="https://colab.research.google.com/github/leilacielok/market_basket_analysis/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorithms for Massive Data - Market-Basket Analysis

In [1]:
import os
import pandas as pd

from google.colab import userdata
from itertools import combinations
from collections import Counter

In [12]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")


# SOSTITUIRE CON: Insert your Kaggle credentials before running
# os.environ['KAGGLE_USERNAME'] = "xxxx"
# os.environ['KAGGLE_KEY'] = "xxxx"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o -q imdb-dataset-of-top-1000-movies-and-tv-shows.zip -d imdb_data

print("Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded and extracted.


In [48]:
csv_path = os.path.join(
    "imdb_data",
    "imdb_top_1000.csv"
)

df = pd.read_csv(csv_path)

In [49]:
# Minimum support threshold
MIN_SUPPORT_RATIO = 0.005
MIN_SUPPORT_COUNT = int(len(df) * MIN_SUPPORT_RATIO)

print("Minimum support count:", MIN_SUPPORT_COUNT)

Minimum support count: 5


In [50]:
# Basket construction
STAR_COLS = ["Star1", "Star2", "Star3", "Star4"]

baskets = []

for _, row in df.iterrows():
    basket = []

    for col in STAR_COLS:
        actor = row[col]
        basket.append(actor)

    baskets.append(set(basket))

print("Number of baskets:", len(baskets))
print("Example basket:", baskets[0])

Number of baskets: 1000
Example basket: {'Morgan Freeman', 'Bob Gunton', 'Tim Robbins', 'William Sadler'}


## A-Priori Algorithm

In [60]:
def apriori(baskets, support_threshold):

    # FIRST PASS
    # Translate item names into integers

    names2int = {}
    int2names = {}
    item_id = 0

    for basket in baskets:
        for item in basket:
            if item not in names2int:
                names2int[item] = item_id
                int2names[item_id] = item
                item_id += 1

    # Use integers as indexes in the counts array
    counts = [0] * len(names2int)

    for basket in baskets:
        for item in basket:
            item_integer = names2int[item]
            counts[item_integer] += 1

    # IN-BETWEEN PASSES
    # Create frequent-items table

    frequent_items_table = [0] * len(counts)
    frequent_singletons = {}

    new_number = 1
    newint2name = {}

    for item_integer, count in enumerate(counts):
        if count >= support_threshold:
            frequent_items_table[item_integer] = new_number
            newint2name[new_number] = int2names[item_integer]
            frequent_singletons[frozenset([int2names[item_integer]])] = count
            new_number += 1

    # SECOND PASS
    # Count all pairs of frequent items

    pair_counts = {}

    for basket in baskets:

        # a. Find frequent items in the basket
        frequent_items_in_basket = []

        for item in basket:
            old_integer = names2int[item]
            new_integer = frequent_items_table[old_integer]

            if new_integer != 0:
                frequent_items_in_basket.append(new_integer)

        # b. Generate all pairs of frequent items
        for i in range(len(frequent_items_in_basket)):
            for j in range(i + 1, len(frequent_items_in_basket)):

                item1 = frequent_items_in_basket[i]
                item2 = frequent_items_in_basket[j]
                pair = tuple(sorted((item1, item2)))

                if pair not in pair_counts:
                    pair_counts[pair] = 0

                pair_counts[pair] += 1

    frequent_pairs = {}

    for pair, count in pair_counts.items():
        if count >= support_threshold:
            actor1 = newint2name[pair[0]]
            actor2 = newint2name[pair[1]]

            frequent_pairs[frozenset([actor1, actor2])] = count

     # THIRD PASS
     # Count triples generated from frequent pairs

    triple_counts = {}

    frequent_pair_sets = set(frequent_pairs.keys())

    for basket in baskets:

        frequent_items_in_basket = []

        for item in basket:
            old_integer = names2int[item]
            new_integer = frequent_items_table[old_integer]

            if new_integer != 0:
                frequent_items_in_basket.append(item)

        for i in range(len(frequent_items_in_basket)):
            for j in range(i + 1, len(frequent_items_in_basket)):
                for k in range(j + 1, len(frequent_items_in_basket)):

                    triple = frozenset([
                        frequent_items_in_basket[i],
                        frequent_items_in_basket[j],
                        frequent_items_in_basket[k]
                    ])

                    pair1 = frozenset([frequent_items_in_basket[i], frequent_items_in_basket[j]])
                    pair2 = frozenset([frequent_items_in_basket[i], frequent_items_in_basket[k]])
                    pair3 = frozenset([frequent_items_in_basket[j], frequent_items_in_basket[k]])

                    if pair1 in frequent_pair_sets and pair2 in frequent_pair_sets and pair3 in frequent_pair_sets:
                        if triple not in triple_counts:
                            triple_counts[triple] = 0

                        triple_counts[triple] += 1

    frequent_triples = {}

    for triple, count in triple_counts.items():
        if count >= support_threshold:
            frequent_triples[triple] = count

    frequent_itemsets = {}

    frequent_itemsets[1] = frequent_singletons

    if frequent_pairs:
        frequent_itemsets[2] = frequent_pairs
    if frequent_triples:
        frequent_itemsets[3] = frequent_triples

    return frequent_itemsets

In [61]:
# Display frequent itemsets
frequent_itemsets = apriori(
    baskets,
    MIN_SUPPORT_COUNT
)

for size, itemsets in frequent_itemsets.items():
    print("\n" + "=" * 50)
    print(f"Frequent itemsets of size {size}")
    print("=" * 50)

    sorted_itemsets = sorted(
        itemsets.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for itemset, count in sorted_itemsets:
        print(set(itemset), "-> support count:", count)


Frequent itemsets of size 1
{'Robert De Niro'} -> support count: 17
{'Tom Hanks'} -> support count: 14
{'Al Pacino'} -> support count: 13
{'Brad Pitt'} -> support count: 12
{'Clint Eastwood'} -> support count: 12
{'Christian Bale'} -> support count: 11
{'Leonardo DiCaprio'} -> support count: 11
{'Matt Damon'} -> support count: 11
{'James Stewart'} -> support count: 10
{'Michael Caine'} -> support count: 9
{'Scarlett Johansson'} -> support count: 9
{'Humphrey Bogart'} -> support count: 9
{'Ethan Hawke'} -> support count: 9
{'Johnny Depp'} -> support count: 9
{'Denzel Washington'} -> support count: 9
{'Harrison Ford'} -> support count: 8
{'Aamir Khan'} -> support count: 8
{'Morgan Freeman'} -> support count: 7
{'Ian McKellen'} -> support count: 7
{'Bruce Willis'} -> support count: 7
{'Edward Norton'} -> support count: 7
{'Toshirô Mifune'} -> support count: 7
{'Russell Crowe'} -> support count: 7
{'Mark Ruffalo'} -> support count: 7
{'Robert Downey Jr.'} -> support count: 7
{'Cary Grant'